# Starter 3: Measuring in different bases

| | |
|---|---|
| **Level** | Introductory |
| **Time** | About 40 minutes |
| **Prerequisites** | Qubits, the Hadamard and S gates, measurement |
| **Default device** | IQM Garnet |
| **Also runs on** | Rigetti Cepheus-1-108Q, AQT IBEX Q1 |
| **Qubits** | 9 |
| **Two-qubit gates** | None |
| **Hardware jobs** | 1 |
| **Approximate cost** | Garnet at 500 shots: about 103 credits. Rigetti: about 10 credits (billed by execution time). |
| **Suggested hand-in** | Your two tables, and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST starter series from qBraid. You may copy, edit and adapt this notebook for your course.*

A measurement asks a qubit one question. The usual question is "0 or 1?", which is a measurement in the **Z basis**. You can ask a different question by rotating the qubit first and then measuring:

| To measure in | apply before measuring |
|---|---|
| Z basis | nothing |
| X basis | H |
| Y basis | S$^\dagger$, then H |

In this notebook you prepare three states and measure each of them in all three bases. A state that gives a certain answer in one basis gives a random answer in the others. That is the qubit version of the uncertainty principle: no single measurement reveals the whole state.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "aws:iqm:qpu:garnet"   # device list and prices: see the README
SHOTS = 500                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

Three states times three bases gives nine combinations. Each runs on its own qubit, so the whole experiment is one hardware job.

| State | Prepared with |
|---|---|
| $\|0\rangle$ | nothing |
| $\|+\rangle$ | H |
| $\|{+i}\rangle$ | H, then S |

In [ ]:
STATES = ["|0>", "|+>", "|+i>"]
BASES = ["Z", "X", "Y"]
N_QUBITS = len(STATES) * len(BASES)

def prepare(qc, state, qubit):
    if state in ("|+>", "|+i>"):
        qc.h(qubit)
    if state == "|+i>":
        qc.s(qubit)

def rotate_to_basis(qc, basis, qubit):
    if basis == "X":
        qc.h(qubit)
    elif basis == "Y":
        qc.sdg(qubit)
        qc.h(qubit)

qc = QuantumCircuit(N_QUBITS)
for i, state in enumerate(STATES):
    for j, basis in enumerate(BASES):
        qubit = 3 * i + j
        prepare(qc, state, qubit)
        rotate_to_basis(qc, basis, qubit)
qc.measure_all()

qc.draw(output="text")

## 2. Ideal simulation

The table shows the probability of the outcome 0 for each state and basis. Each row has one entry of 1 (a certain answer) and two of 0.5 (a coin flip).

In [ ]:
def p0_table(counts):
    table = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            table[i, j] = 1 - prob_one(counts, 3 * i + j, N_QUBITS)
    return table

def show(table, label):
    print(f"{label}: probability of outcome 0")
    print("state    " + "".join(f"{b:>8}" for b in BASES))
    for i, state in enumerate(STATES):
        print(f"{state:<9}" + "".join(f"{table[i, j]:8.3f}" for j in range(3)))
    print()

simulator = AerSimulator()
ideal_counts = simulator.run(qc, shots=SHOTS).result().get_counts()
ideal_table = p0_table(ideal_counts)
show(ideal_table, "Ideal simulation")

## 3. Run on hardware

In [ ]:
N_JOBS = 1

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    job = device.run(qc, shots=SHOTS)
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = job.result().data.get_counts()
    print(f"Received {sum(hw_counts.values())} shots.")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

In [ ]:
tables = [("Ideal simulation", ideal_table)]
if hw_counts:
    hw_table = p0_table(hw_counts)
    show(hw_table, "Hardware")
    tables.append((f"Hardware ({DEVICE_ID})", hw_table))

fig, axes = plt.subplots(1, len(tables), figsize=(4.5 * len(tables), 3.8))
axes = np.atleast_1d(axes)
for ax, (title, table) in zip(axes, tables):
    image = ax.imshow(table, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(3), [f"measure {b}" for b in BASES])
    ax.set_yticks(range(3), STATES)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{table[i, j]:.2f}", ha="center", va="center", color="white")
    ax.set_title(title, fontsize=10)
fig.colorbar(image, ax=axes, label="probability of outcome 0")
plt.show()

## Questions to try

1. On hardware, how far are the "certain" entries from 1, and the "coin flip" entries from 0.5? Which kind of entry is affected more by noise?
2. Using only the Z-basis column, can you tell $|+\rangle$ from $|{+i}\rangle$? Which measurement do you need to distinguish them?
3. Prepare the state $R_y(\pi/3)|0\rangle$ and measure it in all three bases. Use the results to estimate $\langle X\rangle$, $\langle Y\rangle$ and $\langle Z\rangle$, where $\langle Z\rangle = P(0) - P(1)$. The ideal values are $(\sin\frac{\pi}{3}, 0, \cos\frac{\pi}{3})$. Is the length of the measured vector less than 1? What does that tell you about the state on the hardware?